# Recipe Chat Bot
### Start installing needed packages
### mounting your drive if needed


In [ ]:
!pip install -q langchain langchain-community langchain-chroma
!pip install -q langchain_openai langchain-classic
!pip install -q langchain-huggingface huggingface_hub
!pip install -q langchain-text-splitters pypdf pymupdf chromadb
# Force-upgrade bitsandbytes and accelerate to meet minimum version requirements
!pip install -U -q "bitsandbytes>=0.46.1" accelerate

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Load PDFs from Google Drive using PyMuPDF

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
#from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader

PDF_FOLDER = "/content/drive/MyDrive/Recipe_RAG/recipe_pdfs"
DB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"

# 1. Load PDFs from Google Drive using PyMuPDF
# We pass `silent_errors=True` to DirectoryLoader to avoid crash on bad files
loader = PyPDFDirectoryLoader(PDF_FOLDER, glob="**/*.pdf",
                              silent_errors=True)
#loader = DirectoryLoader(
 #   PDF_FOLDER,
 #   glob="**/*.pdf",
  #  loader_cls=PyMuPDFLoader,
  #  show_progress=True,
 #   silent_errors=True
#)
try:
  docs = loader.load()
  print(f"Successfully loaded {len(docs)} pages/documents!")
except Exception as e:
    print(f"An error occurred during loading: {e}")

/tmp/ipykernel_423/1162027745.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFDirectoryLoader


Successfully loaded 1976 pages/documents!


# Split into recipe-sized chunks


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 2. Split into recipe-sized chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1800,
    chunk_overlap=250,
    separators=["\n\n", "\n", ".", " "]
)
chunks = text_splitter.split_documents(docs)
print(f"{len(chunks)} chunks created")

2953 chunks created


#### Print sample chunks to check how they split

In [ ]:
# Inspect Chunk 10 and Chunk 11 (skipping early table-of-contents pages)

for i in range(10, 12):
    print(f"================ CHUNK {i} ================")
    print(f"Length: {len(chunks[i].page_content)} characters")
    print(f"Metadata: {chunks[i].metadata}")
    print("-" * 40)
    print(chunks[i].page_content)
    print("\n")

================ CHUNK 10 ================
Length: 72 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page': 9, 'page_label': '10'}
----------------------------------------
Preparation Time: 10 minutes
clementine, raisin & Goat cheese Sandwich
7


================ CHUNK 11 ================
Length: 1414 characters
Metadata: {'producer': 'Adobe PDF Library 10.0.1', 'creator': 'Adobe InDesign CS6 (Macintosh)', 'creationdate': '2012-09-13T17:06:20-04:00', 'moddate': '2012-09-13T17:10:17-04:00', 'trapped': '/False', 'source': '/content/drive/MyDrive/Recipe_RAG/recipe_pdfs/MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf', 'total_pages': 64, 'page

# Create embeddings and store in ChromaDB
### persist it on the drive

In [ ]:
import os
import torch
from google.colab import userdata

# Store Hugging face read token in the env safely in Colab's left sidebar (Key icon -> Secrets)
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print (f"device we will use = {device}")


device we will use = cpu


   ##### Details at https://huggingface.co/Qwen/Qwen3-Embedding-0.6B
   * Accuracy improves by 1-5% on Qwen3-Embedding models as it is instruction-aware by adding query_encode_kwargs.
   * It helps to treat Queries and Documents Differently as querys are short than chunks.
   * Adding "Instruct: ...\nQuery: " makes LangChain attach it when embedding the user's question and not the chunks.
   * This helps the model to map the question in same vector space neighborhood as full recipe passages.

    

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Pass the detected device into HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
    model_name="Qwen/Qwen3-Embedding-0.6B",
    model_kwargs={"device": device,
        "model_kwargs": {"torch_dtype": torch.float16} # Cuts memory usage in half
    },
    encode_kwargs={
        "batch_size": 32, # Prevents out-of-memory spikes on large datasets
        "normalize_embeddings": True
    },
    # Read details in the text cell above
    query_encode_kwargs={
        "prompt": "Instruct: Given a cooking question or dish request, retrieve relevant recipe passages with ingredients and instructions.\nQuery: ",
        "normalize_embeddings": True
    }
)

# uncomment following code if need to regenerate the vectors
# remove the old saved from drive else double chunks will be created
"""
# 3. Create and save Chroma DB permanently to Google Drive
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_FOLDER
)
print(f"Saved {len(chunks)} recipe chunks to Google Drive!")
"""

# uncomment following code to load vectors from saved drive

# 3. Load saved Chroma DB from Google Drive
DB_FOLDER = "/content/drive/MyDrive/Recipe_RAG/chroma_recipes_db"

vectorstore = Chroma(
    persist_directory=DB_FOLDER,
    embedding_function=embeddings
)

# 3. Verify that all chunks loaded properly
print(f"Loaded {vectorstore._collection.count()} chunks from Google Drive!")


Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loaded 2953 chunks from Google Drive!


# Build RAG chain for QA with the bot

In [ ]:
import torch
from langchain_huggingface import HuggingFacePipeline, ChatHuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA

# get the runtime device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_id = "Qwen/Qwen2.5-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)

if device == "cuda":
    # Runs on Colab T4 GPU or paid Hugging Face GPU Spaces
    # Configure 4-bit quantization to fit the model in limited VRAM
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quantization_config,
        device_map="auto"
    )
else:
    # Runs on Hugging Face's free CPU Basic tier (16 GB RAM)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.bfloat16,
        device_map="cpu"
    )

# Ensure Qwen stops immediately at <|im_end|> instead of generating extra tokens
stop_ids = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|im_end|>")
]

# 1. Create native Transformers Pipeline
hf_pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=600,
    do_sample=False,           # Greedy decoding
    repetition_penalty=1.1,
    eos_token_id=stop_ids,
    pad_token_id=tokenizer.eos_token_id,
    return_full_text=False
)

# 2. Wrap in LangChain's HuggingFacePipeline first!
local_llm = HuggingFacePipeline(pipeline=hf_pipe)

# 3. Now wrap in ChatHuggingFace so it formats as chat messages properly
llm = ChatHuggingFace(llm=local_llm)

# 4. Define the custom cooking PromptTemplate (must use {context} and {question})
recipe_template = """You are an expert chef and helpful recipe assistant.
Answer the user's question using ONLY the recipe context provided below.
Do not invent ingredients or steps that are not in the context.
If the requested recipe is not in the context, reply: "I couldn't find that recipe in your uploaded cookbooks."

When providing a recipe, format your response clearly as:
- **Recipe Name**
- **Prep / Cook Time** (if mentioned)
- **Ingredients** (bulleted list)
- **Instructions** (numbered steps)

Recipe Context:
{context}

User Question: {question}"""

QA_PROMPT = PromptTemplate(
    template=recipe_template,
    input_variables=["context", "question"]
)

# 5. Connect the custom prompt into RetrievalQA via chain_type_kwargs
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k": 4}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": QA_PROMPT}
)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

### Query the recipe bot with a sample question

In [ ]:
# 6. Test your recipe bot

result = qa_chain.invoke({"query": "Give me a bread recipe from the loaded PDFs."})
print("🤖 Bot Response:\n")
print(result["result"])
#print("\nSources used:")
#for doc in result["source_documents"]:
#    print(f"- File: {doc.metadata.get('source')} | Page: {doc.metadata.get('page')}")

🤖 Bot Response:

**Title: Easy Wheat And Honey Bread**

**Servings:** 1

**Ingredients:**
- 1 packet active dry yeast
- 2 tablespoons honey
- 3/4 cup whole wheat flour
- 2 1/2 cups bread flour
- 1 egg, beaten
- 1 teaspoon salt
- 1 cup plus 1 tablespoon warm water

**Instructions:**
1. In a large bowl, combine the yeast and 1 tablespoon of the warm water. Stir gently and let sit until foamy, about 10 minutes.
2. In another bowl, mix the honey, 1 cup of the bread flour, and the egg. Set aside.
3. In a mixing bowl, combine the remaining 1 1/2 cups of bread flour, the salt, and the yeast mixture. Mix until just combined.
4. Gradually add the reserved honey mixture to the flour mixture, stirring until a soft dough forms.
5. Transfer the dough to a greased 1-quart baking dish. Cover with buttered wax paper and a towel. Let rise in a warm place away from drafts until doubled in bulk, about 1 hour.
6. Preheat the oven to 350°F (175°C). Bake for 60 minutes or until the bread sounds hollow when 

In [7]:
# 5. Query!
result = qa_chain.invoke(
    "How do you make a Clementine, Raisin & Goat Cheese Sandwich?"
)
print("🤖 Bot Response:\n")
print(result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"  Source: {doc.metadata.get('source', 'Unknown').split('/')[-1]}, p.{doc.metadata.get('page', 'Unknown')}")

🤖 Bot Response:

**Recipe Name:** Clementine, Raisin & Goat Cheese Sandwich

**Prep / Cook Time:** 10 minutes

**Ingredients:**
- 4 Dandy clementine oranges
- 1 cup Sun-Maid raisins
- 4 oz (1 cup) crumbled goat cheese (or feta cheese)
- 3 tbsp chopped walnuts
- 2 tbsp fat-free yogurt, plain

**Instructions:**
1. Peel clementine oranges and split each in half.
2. With a sharp serrated knife, chop the orange halves into small pieces.
3. In a medium bowl, mix raisins, cheese, walnuts, yogurt, and mayonnaise.
4. Add chopped clementine oranges and mix gently.
5. Lay out bread slices.
6. Spread ¼ salad mix on 4 of the bread slices.
7. Add two lettuce leaves to each sandwich; top with 2nd bread slice.

Serves 4.

Sources:
  Source: MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf, p.9
  Source: MyPlate_RecipeBook2012_FINAL_SinglePages-5B8E06286B084B78B2EA5174E1711C5F.pdf, p.10
  Source: Cooking - 500 recipes for bread.pdf, p.112
  Source: Leaves From The Tree of L

# Set up interactive demo using Gradio in colab

In [8]:
!pip install -q gradio

import os
import gradio as gr

def chat_with_chef(message, history):
    # Query your existing RetrievalQA chain
    response = qa_chain.invoke({"query": message})
    answer = response["result"].strip()

    # Format the PDF sources cleanly below the recipe
    sources = []
    for doc in response.get("source_documents", []):
        file_name = os.path.basename(doc.metadata.get("source", "Unknown PDF"))
        page_num = doc.metadata.get("page", 0) + 1
        source_tag = f"{file_name} (Page {page_num})"
        if source_tag not in sources:
            sources.append(source_tag)

    if sources:
        answer += "\n\n---\n**Sources:** " + ", ".join(sources)

    return answer

# Launch the Gradio ChatInterface inside Colab
demo = gr.ChatInterface(
    fn=chat_with_chef,
    title="🍳 Cookbook RAG Recipe Assistant",
    description="Ask for recipes, ingredients, or cooking instructions from uploaded PDF cookbooks.",
    examples=[
        "Give me a adrak chai recipe from the loaded PDFs.",
        "What vegetarian dinner recipes do you have?",
        "How do I make pasta sauce?"
    ]
)

demo.launch(share=True, debug=False)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9af44740d35c63eb8f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
